# 🎓 Day 14 — FINAL DAY: Bayesian HPO, Entity Embeddings, Multi-Task & Grand Finale
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Bayesian HPO with Gaussian Process | ~12 sec |
| 2 | 10:30–11:00 AM | Entity Embeddings & Tabular Deep Learning | ~5 sec |
| 3 | 11:00 AM–1:00 PM | Multi-Task Learning + Grand Finale Project | ~18 sec |
| 4 | 1:00–2:00 PM | Lunch & 14-Day Reflection | — |
| 5 | 2:00–4:00 PM | 14-Day Comprehensive Production Pipeline | ~20 sec |

> **Zero downloads. Pure numpy + sklearn + optuna + scipy.**
---

In [ ]:
# !pip install scikit-learn optuna matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm
import warnings, time, json, hashlib
from datetime import datetime
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, load_wine, load_iris, make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import sklearn
print('All imports OK')
print(f'   sklearn : {sklearn.__version__}')
print(f'   optuna  : {optuna.__version__}')

---
## Session 1 — Bayesian Hyperparameter Optimization (GP-Based)
**Time:** 9:00–10:30 AM | **Run time: ~12 sec**

### Key concepts
- Gaussian Process: probabilistic surrogate model over hyperparameter space
- Expected Improvement: acquisition function balancing exploration and exploitation
- Sequential model-based optimization outperforms random and grid search

In [ ]:
# 1.1  Gaussian Process Surrogate
def rbf_kernel(x1, x2, length_scale=0.3, signal_var=1.0):
    """RBF (squared exponential) kernel."""
    x1 = np.atleast_2d(x1).T if x1.ndim == 1 else x1
    x2 = np.atleast_2d(x2).T if x2.ndim == 1 else x2
    diff = x1[:, None, :] - x2[None, :, :]
    return signal_var**2 * np.exp(-0.5 * np.sum((diff / length_scale)**2, axis=-1))

def gp_predict(X_obs, y_obs, X_new, noise_var=1e-5):
    """Gaussian Process posterior mean and std."""
    K_obs   = rbf_kernel(X_obs, X_obs) + noise_var * np.eye(len(X_obs))
    K_cross = rbf_kernel(X_obs, X_new)
    K_new   = rbf_kernel(X_new, X_new)

    L    = np.linalg.cholesky(K_obs + 1e-6*np.eye(len(K_obs)))
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_obs))
    mu    = K_cross.T @ alpha

    v    = np.linalg.solve(L, K_cross)
    cov  = K_new - v.T @ v
    std  = np.sqrt(np.maximum(0, np.diag(cov)))
    return mu, std

# Visualise GP on a 1D toy function
true_fn  = lambda x: np.sin(3*x) * x + 0.5*x**2

np.random.seed(42)
X_obs = np.array([0.1, 0.4, 0.7, 1.2, 1.8]).reshape(-1, 1)
y_obs = true_fn(X_obs.ravel()) + np.random.normal(0, 0.05, len(X_obs))

X_new = np.linspace(0, 2, 200).reshape(-1, 1)
mu, std = gp_predict(X_obs, y_obs, X_new)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(X_new, true_fn(X_new.ravel()), 'k--', linewidth=1.5, alpha=0.5, label='True function')
ax.plot(X_new, mu, color='#534AB7', linewidth=2, label='GP mean')
ax.fill_between(X_new.ravel(), mu-2*std, mu+2*std, alpha=0.2, color='#534AB7', label='95% CI')
ax.scatter(X_obs, y_obs, color='#D85A30', s=80, zorder=5, label='Observations')
ax.set_title('Gaussian Process Posterior (surrogate model)')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 1.2  Expected Improvement Acquisition Function
def expected_improvement(mu, std, y_best, xi=0.01):
    """EI: expected amount by which new point improves on best seen."""
    z   = (mu - y_best - xi) / (std + 1e-9)
    ei  = (mu - y_best - xi) * norm.cdf(z) + std * norm.pdf(z)
    return np.maximum(0, ei)

def upper_confidence_bound(mu, std, kappa=2.0):
    """UCB: optimistic estimate (mean + kappa * uncertainty)."""
    return mu + kappa * std

y_best = y_obs.max()
ei_vals  = expected_improvement(mu, std, y_best)
ucb_vals = upper_confidence_bound(mu, std)

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# Top: GP posterior
axes[0].plot(X_new, true_fn(X_new.ravel()), 'k--', linewidth=1.5, alpha=0.5, label='True fn')
axes[0].plot(X_new, mu, color='#534AB7', linewidth=2, label='GP mean')
axes[0].fill_between(X_new.ravel(), mu-2*std, mu+2*std, alpha=0.15, color='#534AB7')
axes[0].scatter(X_obs, y_obs, color='#D85A30', s=80, zorder=5)
next_ei = X_new.ravel()[ei_vals.argmax()]
axes[0].axvline(next_ei, color='#1D9E75', linestyle='--', linewidth=2,
                label=f'Next query (EI): x={next_ei:.3f}')
axes[0].set_title('GP Posterior'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Bottom: acquisition function
axes[1].plot(X_new.ravel(), ei_vals,  color='#1D9E75', linewidth=2, label='Expected Improvement')
axes[1].plot(X_new.ravel(), ucb_vals/ucb_vals.max(), color='#D85A30', linewidth=2,
             linestyle='--', label='UCB (normalised)')
axes[1].axvline(next_ei, color='#1D9E75', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('x'); axes[1].set_ylabel('Acquisition value')
axes[1].set_title('Acquisition Functions'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'EI suggests querying at x={next_ei:.4f}')

In [ ]:
# 1.3  Full Bayesian HPO Loop Using Optuna (GP backend)
X_hpo, y_hpo = load_wine(return_X_y=True)

def objective_wine(trial):
    model = RandomForestClassifier(
        n_estimators      = trial.suggest_int('n_estimators', 20, 200),
        max_depth         = trial.suggest_int('max_depth', 2, 15),
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10),
        max_features      = trial.suggest_float('max_features', 0.3, 1.0),
        random_state      = 42
    )
    return cross_val_score(model, X_hpo, y_hpo, cv=5, scoring='accuracy').mean()

# Compare: Random search vs Bayesian (TPE)
study_bayes  = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
study_random = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.RandomSampler(seed=42))

N_TRIALS = 40
study_bayes.optimize(objective_wine,  n_trials=N_TRIALS, show_progress_bar=False)
study_random.optimize(objective_wine, n_trials=N_TRIALS, show_progress_bar=False)

print(f'Bayesian (TPE) best: {study_bayes.best_value:.4f}  params={study_bayes.best_params}')
print(f'Random search best : {study_random.best_value:.4f}')

In [ ]:
# 1.4  HPO Convergence Comparison
bayes_best_so_far  = [max([t.value for t in study_bayes.trials[:i+1]])
                       for i in range(N_TRIALS)]
random_best_so_far = [max([t.value for t in study_random.trials[:i+1]])
                       for i in range(N_TRIALS)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, N_TRIALS+1), bayes_best_so_far,  'o-', color='#534AB7',
        linewidth=2, markersize=4, label='Bayesian (TPE)')
ax.plot(range(1, N_TRIALS+1), random_best_so_far, 's-', color='#D85A30',
        linewidth=2, markersize=4, linestyle='--', label='Random search')
ax.set_xlabel('Number of trials'); ax.set_ylabel('Best CV accuracy so far')
ax.set_title('Bayesian vs Random HPO — Wine Dataset')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

regret_bayes  = max(bayes_best_so_far)  - np.array(bayes_best_so_far)
regret_random = max(random_best_so_far) - np.array(random_best_so_far)
print(f'Cumulative regret — Bayesian: {regret_bayes.sum():.4f}  '
      f'Random: {regret_random.sum():.4f}  '
      f'Efficiency gain: {regret_random.sum()/regret_bayes.sum():.2f}x')

---
## Session 2 — Entity Embeddings & Tabular Deep Learning
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  Entity Embedding System
rng = np.random.default_rng(42)

categorical_meta = {
    'city'         : 150,   # 150 unique cities
    'product_cat'  : 800,   # 800 product categories
    'user_segment' : 12,    # 12 user segments
    'day_of_week'  : 7,     # 7 days
    'hour_of_day'  : 24,    # 24 hours
}

class EmbeddingTable:
    def __init__(self, n_categories, embed_dim):
        self.E = rng.normal(0, 0.01, (n_categories, embed_dim))
        self.n = n_categories
        self.d = embed_dim

    def lookup(self, indices):
        return self.E[indices]

    def similarity(self, idx_a, idx_b):
        a = self.E[idx_a]; b = self.E[idx_b]
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

# Build embedding tables
embeds = {}
for name, n_cats in categorical_meta.items():
    dim       = min(50, max(4, int(np.ceil(n_cats / 2))))
    embeds[name] = EmbeddingTable(n_cats, dim)
    reduction = (1 - dim/n_cats) * 100
    print(f'{name:15s}: {n_cats:4d} cats → {dim:3d}D  '
          f'({dim*n_cats:6d} params  vs {n_cats**2:7d} one-hot params  {reduction:.0f}% smaller)')

In [ ]:
# 2.2  Feature Crossing + Wide & Deep Concept
def build_tabular_features(city_id, product_id, segment_id, hour, day,
                            numeric_feats):
    """Combine entity embeddings + numeric features + crossed features."""
    # Entity embeddings
    city_emb   = embeds['city'].lookup(city_id)
    prod_emb   = embeds['product_cat'].lookup(product_id)
    seg_emb    = embeds['user_segment'].lookup(segment_id)
    hour_emb   = embeds['hour_of_day'].lookup(hour)
    day_emb    = embeds['day_of_week'].lookup(day)

    # Feature crosses (Hadamard product = element-wise interaction)
    city_prod_cross  = city_emb[:prod_emb.shape[0]] * prod_emb  # city × product
    seg_hour_cross   = seg_emb[:hour_emb.shape[0]] * hour_emb   # segment × hour

    # Wide features: raw numeric + one-hot of high-freq categories
    wide = np.concatenate([numeric_feats, [hour/24.0, day/7.0]])

    # Deep features: all embeddings concatenated
    deep = np.concatenate([city_emb, prod_emb, seg_emb, hour_emb, day_emb,
                           city_prod_cross, seg_hour_cross])

    # Combined: wide + deep
    return np.concatenate([wide, deep])

# Sample usage
sample = build_tabular_features(
    city_id=42, product_id=199, segment_id=3,
    hour=14, day=2,
    numeric_feats=np.array([29.99, 0.85, 3.2])
)
print(f'Wide+Deep feature vector dimension: {len(sample)}')
print(f'  Numeric (wide)       : 5 features')
print(f'  All embeddings (deep): {len(sample)-5} features')

In [ ]:
# 2.3  TabNet-Style Sequential Attention (Concept)
class TabNetAttention:
    """Simplified sequential attention: select features per step."""
    def __init__(self, n_features, n_steps=3):
        self.n_features = n_features
        self.n_steps    = n_steps
        rng2 = np.random.default_rng(0)
        # Attention weights per step (learned in real TabNet)
        self.attn_weights = rng2.normal(0, 0.1, (n_steps, n_features))

    def forward(self, x):
        """Return attended features across all steps."""
        step_outputs  = []
        used_features = np.zeros(self.n_features)  # track feature usage

        for step in range(self.n_steps):
            # Softmax attention — sparse feature selection
            raw_attn  = self.attn_weights[step] * (1 - used_features)
            attn      = np.exp(raw_attn) / (np.exp(raw_attn).sum() + 1e-9)
            used_features += attn  # penalise reuse
            attended      = attn * x
            step_outputs.append(attended)

        return np.vstack(step_outputs), attn  # (n_steps, n_features)

# Demo
rng_d = np.random.default_rng(42)
x_sample = rng_d.normal(0, 1, 20)  # 20 features
tabnet   = TabNetAttention(n_features=20, n_steps=3)
step_out, final_attn = tabnet.forward(x_sample)

print('TabNet attention weights per step:')
for s in range(3):
    top_feat = np.argsort(step_out[s])[::-1][:3]
    print(f'  Step {s+1}: top features = {list(top_feat)}  '
          f'max_attn = {step_out[s].max():.4f}')

---
## Session 3 — Multi-Task Learning
**Time:** 11:00 AM–1:00 PM | **Run time: ~18 sec**

In [ ]:
# 3.1  Multi-Task Learning: Shared Representation
rng = np.random.default_rng(42)
N   = 1500

# 3 related classification tasks sharing the same input
X_mtl = rng.normal(0, 1, (N, 15))
y1 = ((X_mtl[:,0] + X_mtl[:,1] + rng.normal(0,0.3,N)) > 0).astype(int)  # fraud
y2 = ((X_mtl[:,2] - X_mtl[:,3] + rng.normal(0,0.3,N)) > 0).astype(int)  # churn
y3 = ((X_mtl[:,4] + X_mtl[:,5] + X_mtl[:,6] + rng.normal(0,0.4,N)) > 0.5).astype(int)  # upsell

X_tr, X_te = X_mtl[:1200], X_mtl[1200:]
y1_tr, y1_te = y1[:1200], y1[1200:]
y2_tr, y2_te = y2[:1200], y2[1200:]
y3_tr, y3_te = y3[:1200], y3[1200:]

# Approach 1: Single-task (separate model per task)
st_results = []
for task, y_tr, y_te in [('Fraud',y1_tr,y1_te),('Churn',y2_tr,y2_te),('Upsell',y3_tr,y3_te)]:
    m  = RandomForestClassifier(60, random_state=42).fit(X_tr, y_tr)
    st_results.append({'task':task, 'acc':round(m.score(X_te,y_te),4),
                       'auc':round(roc_auc_score(y_te,m.predict_proba(X_te)[:,1]),4)})
print('Single-task results:')
for r in st_results: print(f'  {r["task"]:8s}: acc={r["acc"]:.4f}  auc={r["auc"]:.4f}')

In [ ]:
# 3.2  Multi-Task: Shared Features → Task-Specific Heads
from sklearn.decomposition import PCA

# Step 1: Learn shared representation (PCA as proxy for shared encoder)
scaler_mtl = StandardScaler().fit(X_tr)
X_tr_sc    = scaler_mtl.transform(X_tr)
X_te_sc    = scaler_mtl.transform(X_te)

pca_shared = PCA(n_components=8, random_state=42).fit(X_tr_sc)
X_tr_shared = pca_shared.transform(X_tr_sc)
X_te_shared = pca_shared.transform(X_te_sc)

# Step 2: Task-specific heads (lightweight classifiers on shared features)
mt_results = []
for task, y_tr, y_te in [('Fraud',y1_tr,y1_te),('Churn',y2_tr,y2_te),('Upsell',y3_tr,y3_te)]:
    # Also append original features (concat shared + original)
    X_tr_full = np.hstack([X_tr_shared, X_tr_sc])
    X_te_full = np.hstack([X_te_shared, X_te_sc])
    m  = LogisticRegression(max_iter=500).fit(X_tr_full, y_tr)
    mt_results.append({'task':task, 'acc':round(m.score(X_te_full,y_te),4),
                       'auc':round(roc_auc_score(y_te,m.predict_proba(X_te_full)[:,1]),4)})

print('Multi-task (shared encoder) results:')
for r in mt_results: print(f'  {r["task"]:8s}: acc={r["acc"]:.4f}  auc={r["auc"]:.4f}')

# Compare improvement
print('\nImprovement from shared features:')
for st, mt in zip(st_results, mt_results):
    delta = mt['auc'] - st['auc']
    print(f'  {st["task"]:8s}: AUC {st["auc"]} → {mt["auc"]}  ({delta:+.4f})')

In [ ]:
# 3.3  Uncertainty Weighting for Multi-Task Loss
# Uncertainty weighting: each task weight = 1/(2*sigma_i^2) + log(sigma_i)
# sigma_i is learned per task — tasks with high uncertainty get lower weight

def uncertainty_weighting(task_losses, n_epochs=50):
    """Simulate learning task weights via uncertainty."""
    rng_uw = np.random.default_rng(0)
    log_sigmas = rng_uw.normal(0, 0.1, len(task_losses))  # initialise
    history    = []

    for epoch in range(n_epochs):
        sigmas  = np.exp(log_sigmas)
        weights = 1.0 / (2 * sigmas**2)
        # Weighted combined loss
        combined = np.sum(weights * task_losses + np.log(sigmas))
        # Gradient step on log_sigmas (simplified)
        grad_log_sigma = -task_losses / sigmas**2 + 1.0
        log_sigmas     -= 0.01 * grad_log_sigma
        history.append(combined)

    return np.exp(log_sigmas), history

# Simulated task losses (lower = easier task)
task_losses = np.array([0.35, 0.22, 0.41])   # fraud, churn, upsell
sigmas, loss_hist = uncertainty_weighting(task_losses)
weights_norm = (1/(2*sigmas**2)) / (1/(2*sigmas**2)).sum()

print('Uncertainty-weighted task contributions:')
for task, s, w in zip(['Fraud','Churn','Upsell'], sigmas, weights_norm):
    print(f'  {task:8s}: sigma={s:.4f}  weight={w:.4f}  '
          f'({"high" if w>0.4 else "medium" if w>0.25 else "low"} priority)')

---
## Lunch Break / 14-Day Reflection — 1:00–2:00 PM

**14 skills you now own:**

1. EDA → cleaning → feature engineering pipeline
2. Supervised ML: regression, classification, ensembles
3. Deep learning: MLP, CNN, backprop, PyTorch
4. Model deployment: FastAPI, Docker, canary
5. Recommender systems: CF, SVD, RAG
6. Causal inference: IPW, DiD, propensity scores
7. Multi-armed bandits: ε-greedy, UCB, Thompson
8. Federated learning + differential privacy
9. Active learning: entropy, margin, QBC
10. Graph ML: centrality features, community detection
11. SHAP, PDP, ICE, global surrogate explainability
12. Bias audit, model card, audit trail
13. Bayesian HPO, optimizers, matrix methods
14. Multi-task learning, entity embeddings
---

## Session 5 — 14-Day Grand Finale: Full Production ML System
**Time:** 2:00–4:00 PM | **Run time: ~20 sec**

Every major concept from all 14 days integrated into one pipeline.

In [ ]:
# 5.1  Generate Final Competition Dataset
rng = np.random.default_rng(0)
N_FINAL = 2000

df_final = pd.DataFrame({
    'age'          : rng.integers(18, 75, N_FINAL),
    'income'       : rng.exponential(45000, N_FINAL).round(0),
    'tenure_months': rng.integers(1, 120, N_FINAL),
    'num_products'  : rng.integers(1, 8, N_FINAL),
    'support_calls' : rng.integers(0, 15, N_FINAL),
    'late_payments' : rng.integers(0, 6, N_FINAL),
    'credit_score'  : rng.normal(680, 80, N_FINAL).round(0),
    'contract_type' : rng.choice([0,1,2], N_FINAL),
    'gender'        : rng.choice([0,1], N_FINAL),   # protected attribute
    'region'        : rng.choice([0,1,2,3], N_FINAL),
})

# Inject missing values
df_final.loc[rng.choice(N_FINAL, 60, replace=False), 'income']       = np.nan
df_final.loc[rng.choice(N_FINAL, 40, replace=False), 'credit_score'] = np.nan

# Target: churn
logit = (-0.04*df_final['tenure_months'] + 0.12*df_final['support_calls']
         + 0.20*df_final['late_payments'] - 0.30*df_final['contract_type']
         + 0.001*df_final['income'].fillna(45000)/1000
         + rng.normal(0, 0.5, N_FINAL))
df_final['churn'] = (logit > 0.3).astype(int)

print(f'Final dataset: {df_final.shape}')
print(f'Missing: income={df_final.income.isnull().sum()}  credit={df_final.credit_score.isnull().sum()}')
print(f'Churn rate: {df_final.churn.mean():.2%}')

In [ ]:
# 5.2  Data Quality + Feature Engineering
# Step 1: Fill missing values
df_clean = df_final.copy()
df_clean['income']       = df_clean['income'].fillna(df_clean['income'].median())
df_clean['credit_score'] = df_clean['credit_score'].fillna(df_clean['credit_score'].median())

# Step 2: Engineer features
df_clean['income_log']         = np.log1p(df_clean['income'])
df_clean['support_per_tenure'] = df_clean['support_calls'] / (df_clean['tenure_months']+1)
df_clean['credit_income_ratio']= df_clean['credit_score'] / (df_clean['income']/1000+1)
df_clean['high_risk']          = ((df_clean['support_calls']>=8) &
                                   (df_clean['late_payments']>=3)).astype(int)
df_clean['lifetime_value']     = df_clean['income_log'] * df_clean['tenure_months']

FEATURES = [c for c in df_clean.columns
            if c not in ['churn','gender','region']]

X_f = df_clean[FEATURES].values
y_f = df_clean['churn'].values
protected = df_clean['gender'].values

print(f'Feature matrix: {X_f.shape}')
print(f'New features: {[c for c in FEATURES if c not in df_final.columns]}')

In [ ]:
# 5.3  Bayesian HPO on Final Dataset
X_tr_f, X_te_f, y_tr_f, y_te_f, g_tr_f, g_te_f = train_test_split(
    X_f, y_f, protected, test_size=0.2, random_state=42, stratify=y_f
)

def objective_final(trial):
    m = GradientBoostingClassifier(
        n_estimators   = trial.suggest_int('n_estimators', 50, 200),
        learning_rate  = trial.suggest_float('lr', 0.01, 0.3, log=True),
        max_depth      = trial.suggest_int('depth', 2, 6),
        subsample      = trial.suggest_float('subsample', 0.6, 1.0),
        random_state   = 42
    )
    return cross_val_score(m, X_tr_f, y_tr_f, cv=3, scoring='roc_auc').mean()

study_final = optuna.create_study(direction='maximize',
                                   sampler=optuna.samplers.TPESampler(seed=42))
study_final.optimize(objective_final, n_trials=30, show_progress_bar=False)

best_params = study_final.best_params
print(f'Best HPO params: {best_params}')
print(f'Best CV AUC    : {study_final.best_value:.4f}')

In [ ]:
# 5.4  Train Best Model + Full Evaluation
final_model = GradientBoostingClassifier(
    n_estimators  = best_params['n_estimators'],
    learning_rate = best_params['lr'],
    max_depth     = best_params['depth'],
    subsample     = best_params['subsample'],
    random_state  = 42
).fit(X_tr_f, y_tr_f)

final_prob = final_model.predict_proba(X_te_f)[:, 1]
final_pred = (final_prob >= 0.5).astype(int)

final_auc  = roc_auc_score(y_te_f, final_prob)
final_acc  = accuracy_score(y_te_f, final_pred)

print(f'Final Model — AUC: {final_auc:.4f}  Accuracy: {final_acc:.4f}')

# Bias audit
def quick_bias_audit(y_true, y_pred, group, names):
    rows = []
    for g, name in zip(np.unique(group), names):
        m  = group == g
        tp = ((y_pred[m]==1)&(y_true[m]==1)).sum()
        fn = ((y_pred[m]==0)&(y_true[m]==1)).sum()
        fp = ((y_pred[m]==1)&(y_true[m]==0)).sum()
        tn = ((y_pred[m]==0)&(y_true[m]==0)).sum()
        tpr = tp/(tp+fn+1e-9); fpr = fp/(fp+tn+1e-9)
        rows.append({'Group':name,'TPR':round(tpr,4),'FPR':round(fpr,4),
                     'PosRate':round(y_pred[m].mean(),4)})
    df_ba = pd.DataFrame(rows)
    df_ba['DI'] = (df_ba['PosRate']/df_ba['PosRate'].max()).round(4)
    return df_ba

print('\nFairness Audit:')
print(quick_bias_audit(y_te_f, final_pred, g_te_f, ['Female','Male']).to_string(index=False))
di = quick_bias_audit(y_te_f, final_pred, g_te_f, ['Female','Male'])['DI'].min()
print(f'Disparate Impact: {di:.4f}  (>0.8 = fair: {di>=0.8})')

In [ ]:
# 5.5  Production Serving Simulation
import time

# Batch scoring
t0         = time.perf_counter()
batch_probs = final_model.predict_proba(X_te_f)[:, 1]
batch_time  = (time.perf_counter()-t0)*1000
print(f'Batch: {len(X_te_f)} samples in {batch_time:.2f}ms')

# Online serving latency
latencies = []
for i in range(300):
    t0 = time.perf_counter()
    final_model.predict_proba(X_te_f[[i%len(X_te_f)]])
    latencies.append((time.perf_counter()-t0)*1000)
lat = np.array(latencies)
print(f'Online: p50={np.percentile(lat,50):.3f}ms  p95={np.percentile(lat,95):.3f}ms  '
      f'p99={np.percentile(lat,99):.3f}ms')

# Canary simulation
blue_model  = GradientBoostingClassifier(80, random_state=0).fit(X_tr_f, y_tr_f)
blue_auc    = roc_auc_score(y_te_f, blue_model.predict_proba(X_te_f)[:,1])
print(f'\nBlue  (old) AUC: {blue_auc:.4f}')
print(f'Green (new) AUC: {final_auc:.4f}')
promote = final_auc > blue_auc + 0.005
print(f'Promote green to champion: {promote}')

In [ ]:
# 5.6  Final Model Card
print('='*62)
print(' FINAL MODEL CARD — ChurnPredictor v1.0')
print('='*62)
print(f' Date          : {datetime.utcnow().strftime("%Y-%m-%d")}')
print(f' Model         : GradientBoostingClassifier')
print(f' Best params   : n={best_params["n_estimators"]}  '
      f'lr={best_params["lr"]:.4f}  depth={best_params["depth"]}')
print(f' AUC-ROC       : {final_auc:.4f}')
print(f' Accuracy      : {final_acc:.4f}')
print(f' DI ratio      : {di:.4f}  (fair: {di>=0.8})')
print(f' Serving p99   : {np.percentile(lat,99):.3f}ms')
print(f' Features      : {X_f.shape[1]}')
print(f' Training size : {len(X_tr_f)}')
print()
print(' SKILLS DEMONSTRATED IN THIS MODEL:')
skills = ['Data quality check + imputation (Day 10)',
          'Feature engineering: log, ratio, interaction (Days 4,5)',
          'Bayesian HPO with Optuna TPE (Day 14)',
          'Fairness audit: DI ratio, TPR parity (Day 13)',
          'Production latency profiling p50/p95/p99 (Day 9)',
          'Canary deployment decision (Day 10)',
          'Model card generation (Day 13)']
for s in skills:
    print(f'  + {s}')
print('='*62)

---
## 14-Day Internship Summary

| Day | Focus |
|-----|-------|
| 1 | NumPy, Pandas, KNN, train-test split |
| 2 | Regression, ensembles, XGBoost, MLflow |
| 3 | CNNs, transfer learning, BERT, FastAPI, Docker |
| 4 | Pipelines, clustering, PCA, anomaly detection, ARIMA |
| 5 | Q-Learning, feature engineering, Optuna, Kaggle pipeline |
| 6 | Probability, Bayes, backprop, SVMs, custom transformers |
| 7 | Recommenders, A/B testing, fairness, churn prediction |
| 8 | Graph ML, autoencoder anomaly, fraud detection, CI/CD |
| 9 | Embeddings, semantic search, RAG, MLOps dashboard |
| 10 | Data quality, multi-label, SHAP, online learning, canary |
| 11 | Causal inference, survival analysis, bandits, compression |
| 12 | Federated learning, differential privacy, active learning |
| 13 | Optimizers, randomised SVD, PDP/ICE, bias audit, ethics |
| 14 | Bayesian HPO, entity embeddings, multi-task, grand finale |

**You are production-ready.**

In [ ]:
# GRAND FINALE: 14-Day Summary in 14 Lines
print('14-DAY AI/ML INTERNSHIP — GRAND FINALE')
print('='*45)
results_14 = {}
for name, ds_fn in [('Iris',load_iris),('Wine',load_wine),('BreastCancer',load_breast_cancer)]:
    X_d, y_d = ds_fn(return_X_y=True)
    # Bayesian HPO on each
    def obj(trial):
        m = GradientBoostingClassifier(
            n_estimators=trial.suggest_int('n',30,100),
            learning_rate=trial.suggest_float('lr',0.01,0.3,log=True),
            random_state=42)
        return cross_val_score(m, X_d, y_d, cv=3, scoring='accuracy').mean()
    s = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=0))
    s.optimize(obj, n_trials=20, show_progress_bar=False)
    results_14[name] = round(s.best_value, 4)
    print(f'  {name:14s}: Bayesian HPO best acc = {results_14[name]:.4f}')
print('='*45)
print('Internship complete! You are an ML practitioner.')